# Spot the Mask — Training Notebook (Colab-ready)
End-to-end training pipeline. Run each cell in order.


In [ ]:
# ── Setup (run once) ────────────────────────────────────────
# !pip install -q timm albumentations facenet-pytorch omegaconf
import sys, os
sys.path.insert(0, '..')   # adjust if running from project root

In [ ]:
# ── Download data from Zindi ─────────────────────────────────
# Log in to Zindi and download:
#   images.zip → data/raw/images.zip
#   train_labels.csv → data/raw/train_labels.csv
#   sample_submission.csv → data/raw/sample_submission.csv
#
# Then unzip:
# !unzip -q data/raw/images.zip -d data/raw/images/
#
# Verify:
import pandas as pd
from pathlib import Path
train_df = pd.read_csv('../data/raw/train_labels.csv')
print('Train samples:', len(train_df))
print(train_df.head())

In [ ]:
# ── (Optional) Face crop preprocessing ──────────────────────
# This crops faces using MTCNN — recommended for higher AUC.
# Skip if MTCNN is slow or unavailable.
#
# !python -m src.data.preprocess --config configs/config.yaml

In [ ]:
# ── Quick augmentation check ─────────────────────────────────
import cv2, numpy as np, matplotlib.pyplot as plt
from src.data.dataset import get_train_transforms

img = cv2.imread('../data/raw/images/' + train_df.iloc[0]['image'])
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
transform = get_train_transforms(384)

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
axes[0].imshow(img); axes[0].set_title('Original')
for i, ax in enumerate(axes[1:], 1):
    out = transform(image=img)['image']
    # Denormalize for display
    mean = np.array([0.485, 0.456, 0.406])[:,None,None]
    std  = np.array([0.229, 0.224, 0.225])[:,None,None]
    disp = (out.numpy() * std + mean).transpose(1,2,0).clip(0,1)
    ax.imshow(disp); ax.set_title(f'Aug {i}')
for ax in axes: ax.axis('off')
plt.suptitle('Augmentation samples')
plt.tight_layout()
plt.show()

In [ ]:
# ── Train EfficientNet-B4 (fast, good baseline) ───────────────
from src.utils.common import load_config, set_seed
from src.training.trainer import train_kfold

cfg = load_config('../configs/config.yaml')
set_seed(cfg.project.seed)

# Single model training
model_cfg = next(m for m in cfg.models if m['name'] == 'efficientnet_b4')
train_kfold(cfg, model_cfg)

In [ ]:
# ── Train all remaining models ────────────────────────────────
# Warning: this takes several hours on GPU. Run on Colab Pro or Kaggle.
for model_cfg in cfg.models:
    if model_cfg['name'] == 'efficientnet_b4':
        continue   # already trained above
    print(f'\nTraining {model_cfg["name"]}...')
    train_kfold(cfg, model_cfg)

In [ ]:
# ── Analyse OOF results ───────────────────────────────────────
from pathlib import Path
import pandas as pd
from src.utils.common import compute_auc

model_dir = Path('../outputs/models')
for oof_file in sorted(model_dir.glob('*_oof.csv')):
    oof_df = pd.read_csv(oof_file)
    auc = compute_auc(oof_df['target'].values, oof_df['oof_pred'].values)
    print(f'{oof_file.stem:<35}  OOF AUC = {auc:.5f}')

In [ ]:
# ── Generate submission ───────────────────────────────────────
from src.inference.predict import run_inference
run_inference('../configs/config.yaml')
print('Submission saved to outputs/submissions/submission_ensemble.csv')

In [ ]:
# ── Pseudo labeling round 1 ───────────────────────────────────
from src.training.pseudo_label import run_pseudo_labeling
run_pseudo_labeling('../configs/config.yaml', round_num=1)
# Then run inference again:
run_inference('../configs/config.yaml')

In [ ]:
# ── Preview submission ────────────────────────────────────────
sub = pd.read_csv('../outputs/submissions/submission_ensemble.csv')
print(sub.head(10))
print('\nDistribution:')
print(sub['label'].describe())
import matplotlib.pyplot as plt
plt.hist(sub['label'], bins=40, color='#3B8BD4', edgecolor='white')
plt.title('Prediction distribution')
plt.xlabel('Predicted probability')
plt.show()